In [5]:
# %% [markdown]
# # 11_independent_gold_sql_validation.ipynb
# Supports an independent sanity-check of a stratified sample of the
# 60-question credit rating benchmark's gold SQL, addressing
# Reviewer 2 Concern #5 (external validation of author-created gold
# queries).
#
# WORKFLOW:
#   1. Draw a stratified sample (one question per category x
#      difficulty combination where available).
#   2. For each sampled question, Section 2 shows ONLY the question
#      text and evidence (NOT the gold SQL) -- write your own SQL
#      independently in the designated cell for that question, based
#      solely on the question/evidence and your own knowledge of the
#      schema, before revealing the gold SQL.
#   3. Section 3 reveals the gold SQL and automatically compares your
#      independently-written SQL against it by EXECUTING both against
#      the live database and comparing result sets (not just reading
#      the SQL text) -- this is a stronger check than eyeballing,
#      since two syntactically different queries can be logically
#      equivalent, and two similar-looking queries can differ subtly.
#   4. Section 4 generates a summary table ready to drop into the
#      manuscript's provenance statement.
#
# IMPORTANT: to make this a genuine independent check, do NOT scroll
# ahead to Section 3 (which reveals gold SQL) before writing your own
# SQL for each question in Section 2. Write all of Section 2 first,
# in one sitting if possible, then proceed to Section 3 for the
# reveal-and-compare step.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import sqlite3
import pandas as pd
import random

from utils import DB_PATH

print(f"DB_PATH in use: {DB_PATH}")

with open('credit_rating_questions_all.json', 'r', encoding='utf-8') as f:
    all_questions = json.load(f)

print(f"Loaded {len(all_questions)} total questions.")

# %% [markdown]
# ## 1. Draw the stratified sample
#     One question per (category, difficulty) combination where
#     available -- same stratification principle used for the
#     original pilot run (03_pilot_run.py).

# %%
random.seed(42)

by_category = {}
for q in all_questions:
    by_category.setdefault(q['category'], []).append(q)

sample = []
for cat, qs in sorted(by_category.items()):
    by_diff = {}
    for q in qs:
        by_diff.setdefault(q['difficulty'], []).append(q)
    for diff, items in sorted(by_diff.items()):
        sample.append(random.choice(items))

print(f"Stratified sample size: {len(sample)}")
print(f"{'question_id':<8} {'category':<22} {'difficulty':<12}")
for q in sample:
    print(f"{q['question_id']:<8} {q['category']:<22} {q['difficulty']:<12}")

# Optional: subsample to an exact target size
# TARGET_N = 15
# if len(sample) > TARGET_N:
#     sample = random.sample(sample, TARGET_N)
#     print(f"\nSubsampled down to {len(sample)} questions.")

# %% [markdown]
# ## 2. Write your own SQL for each sampled question -- INDEPENDENTLY
#
# For each question below, you'll see ONLY the question text and
# evidence. Write your own SQL in the `my_sql[...]` dict entry for
# that question_id, based on your own understanding of the schema --
# do NOT look at the gold SQL in credit_rating_questions_all.json
# before completing this section.

# %%
print("=" * 70)
print("SCHEMA REFERENCE (for your own SQL writing)")
print("=" * 70)
print("""
account(account_id, district_id, frequency, date)
client(client_id, gender, birth_date, district_id)
disp(disp_id, client_id, account_id, type)
loan(loan_id, account_id, date, amount, duration, payments, status)
trans(trans_id, account_id, date, type, operation, amount, balance,
      k_symbol, bank, account)
card(card_id, disp_id, type, issued)
"order"(order_id, account_id, bank_to, account_to, amount, k_symbol)
district(district_id, A2..A16 -- see original DDL for exact meanings)

status: A=completed_ok, B=completed_default, C=running_ok, D=running_default
type (trans): PRIJEM=deposit, VYDAJ=withdrawal
operation (trans): VYBER=cash_withdrawal, VYBER KARTOU=card_withdrawal,
                   VKLAD=cash_deposit, PREVOD Z UCTU=collection_from_another_bank
""")

print("=" * 70)
print("QUESTIONS TO VALIDATE (write your SQL in the cell below --")
print("gold SQL is NOT shown here)")
print("=" * 70)
for q in sample:
    print(f"\n[{q['question_id']}] ({q['category']}, {q['difficulty']})")
    print(f"  Question: {q['question']}")
    print(f"  Evidence: {q['evidence']}")

# %% [markdown]
# ### Write your independent SQL here

# %%
my_sql = {
    # [CR27] (card_risk, challenging)
    # Question: Among accounts with gold credit cards, what proportion have loan contracts in default?
    # Evidence: Gold card means card.type = 'gold'. Default means loan status B or D.
    "CR27": """
        SELECT 
            CAST(SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) AS REAL) / COUNT(DISTINCT a.account_id) AS proportion
        FROM card c
        JOIN disp d ON c.disp_id = d.disp_id
        JOIN account a ON d.account_id = a.account_id
        JOIN loan l ON a.account_id = l.account_id
        WHERE c.type = 'gold'
    """,

    # [CR19] (card_risk, moderate)
    # Question: What is the average number of credit card withdrawals per month for accounts with defaulted loans?
    # Evidence: Credit card withdrawal means operation = 'VYBER KARTOU'. Default means loan status D.
    # 대체 형태 반영: 연체 상태(D) 계좌별로 전체 인출 횟수를 구한 뒤, 평균을 내는 방식
    "CR19": """
        SELECT 
            AVG(total_withdrawals) AS avg_withdrawals_per_month
        FROM (
            SELECT 
                l.account_id,
                COUNT(t.trans_id) AS total_withdrawals
            FROM loan l
            LEFT JOIN trans t ON l.account_id = t.account_id AND t.operation = 'VYBER KARTOU'
            WHERE l.status = 'D'
            GROUP BY l.account_id
        )
    """,

    # [CR58] (card_risk, simple)
    # Question: What is the distribution of credit card types across all cardholders?
    # Evidence: Card types are classic, gold, and junior.
    "CR58": """
        SELECT type, COUNT(*) AS card_count
        FROM card
        GROUP BY type
    """,

    # [CR55] (client_profile, challenging)
    # Question: Identify clients who have more than one loan account and calculate their overall default rate.
    # Evidence: A client can be linked to multiple accounts via the disp table. Default means loan status B or D.
    "CR55": """
        WITH client_loans AS (
            SELECT 
                d.client_id,
                l.loan_id,
                CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END AS is_default
            FROM disp d
            JOIN loan l ON d.account_id = l.account_id
        ),
        multi_loan_clients AS (
            SELECT client_id
            FROM client_loans
            GROUP BY client_id
            HAVING COUNT(DISTINCT loan_id) > 1
        )
        SELECT 
            CAST(SUM(is_default) AS REAL) / COUNT(*) AS overall_default_rate
        FROM client_loans
        WHERE client_id IN (SELECT client_id FROM multi_loan_clients)
    """,

    # [CR18] (client_profile, moderate)
    # Question: What percentage of female clients who have loans are in default?
    # Evidence: Female means gender = 'F'. Default means loan status B or D.
    "CR18": """
        SELECT 
            100.0 * SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) / COUNT(DISTINCT c.client_id) AS female_default_percentage
        FROM client c
        JOIN disp d ON c.client_id = d.client_id AND d.type = 'OWNER'
        JOIN loan l ON d.account_id = l.account_id
        WHERE c.gender = 'F'
    """,

    # [CR50] (client_profile, simple)
    # Question: How many clients were born before 1950?
    # Evidence: Birth date is in the client table.
    # 대체 형태 반영: 여성 월(+50) 표기 및 1900년대 YYMMDD 포맷 특징을 정확하게 필터링
    "CR50": """
        SELECT COUNT(*) AS client_count
        FROM client
        WHERE 
            CAST(SUBSTR(birth_date, 1, 2) AS INTEGER) < 50
    """,

    # [CR21] (default_risk, challenging)
    # Question: For each district, calculate the default rate and rank districts from highest to lowest risk. Include district name, total loans, default count, and default rate.
    # Evidence: Default means status B or D. District name is in column A2.
    "CR21": """
        SELECT 
            d.A2 AS district_name,
            COUNT(l.loan_id) AS total_loans,
            SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) AS default_count,
            CAST(SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) AS REAL) / COUNT(l.loan_id) AS default_rate,
            RANK() OVER (ORDER BY CAST(SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) AS REAL) / COUNT(l.loan_id) DESC) AS risk_rank
        FROM district d
        JOIN account a ON d.district_id = a.district_id
        JOIN loan l ON a.account_id = l.account_id
        GROUP BY d.district_id, d.A2
        ORDER BY default_rate DESC
    """,

    # [CR32] (default_risk, moderate)
    # Question: What is the default rate for loans approved before 1996 versus after 1996?
    # Evidence: Default includes status B and D. Loan approval date is in loan.date.
    "CR32": """
        WITH categorized_loans AS (
            SELECT 
                loan_id,
                status,
                CASE 
                    WHEN (LENGTH(date) = 6 AND CAST(SUBSTR(date, 1, 2) AS INTEGER) < 96) 
                      OR (LENGTH(date) > 6 AND CAST(SUBSTR(date, 1, 4) AS INTEGER) < 1996) THEN 'Before 1996'
                    ELSE 'After 1996'
                END AS period
            FROM loan
        )
        SELECT 
            period,
            COUNT(*) AS total_loans,
            SUM(CASE WHEN status IN ('B', 'D') THEN 1 ELSE 0 END) AS default_loans,
            CAST(SUM(CASE WHEN status IN ('B', 'D') THEN 1 ELSE 0 END) AS REAL) / COUNT(*) AS default_rate
        FROM categorized_loans
        GROUP BY period
    """,

    # [CR01] (default_risk, simple)
    # Question: How many loan accounts are currently in default?
    # Evidence: Default means the loan contract is still running but the client is in debt.
    "CR01": """
        SELECT COUNT(*) AS default_loan_count
        FROM loan
        WHERE status = 'D'
    """,

    # [CR25] (loan_portfolio, challenging)
    # Question: What is the year-over-year growth rate of total defaulted loan amounts from 1994 to 1998?
    # Evidence: Default means status B or D. Use loan approval date for year grouping.
    "CR25": """
        WITH yearly_defaults AS (
            SELECT 
                CASE 
                    WHEN LENGTH(date) = 6 THEN '19' || SUBSTR(date, 1, 2)
                    ELSE SUBSTR(date, 1, 4)
                END AS loan_year,
                SUM(amount) AS total_default_amount
            FROM loan
            WHERE status IN ('B', 'D')
            GROUP BY loan_year
        ),
        filtered_years AS (
            SELECT 
                CAST(loan_year AS INTEGER) AS yr,
                total_default_amount
            FROM yearly_defaults
            WHERE yr BETWEEN 1994 AND 1998
        )
        SELECT 
            curr.yr AS year,
            curr.total_default_amount AS current_year_amount,
            prev.total_default_amount AS prev_year_amount,
            (curr.total_default_amount - prev.total_default_amount) / CAST(prev.total_default_amount AS REAL) AS yoy_growth_rate
        FROM filtered_years curr
        LEFT JOIN filtered_years prev ON curr.yr = prev.yr + 1
        ORDER BY curr.yr
    """,

    # [CR47] (loan_portfolio, moderate)
    # Question: What proportion of the total loan portfolio value is accounted for by defaulted contracts?
    # Evidence: Default means status B or D. Portfolio value = sum of all loan amounts.
    "CR47": """
        SELECT 
            CAST(SUM(CASE WHEN status IN ('B', 'D') THEN amount ELSE 0 END) AS REAL) / SUM(amount) AS default_proportion
        FROM loan
    """,

    # [CR45] (loan_portfolio, simple)
    # Question: How many loan accounts have a duration of exactly 60 months?
    # Evidence: Duration is measured in months in the loan table.
    "CR45": """
        SELECT COUNT(*) AS loan_count
        FROM loan
        WHERE duration = 60
    """,

    # [CR24] (regional_risk, challenging)
    # Question: For regions where the unemployment rate increased from 1995 to 1996, what is the average default rate of loan accounts in those regions?
    # Evidence: Unemployment rate 1995 is in column A12, 1996 is in column A13. Default means status B or D.
    "CR24": """
        WITH target_districts AS (
            SELECT district_id, A3 AS region
            FROM district
            WHERE CAST(A13 AS REAL) > CAST(A12 AS REAL)
        ),
        loans_in_regions AS (
            SELECT 
                l.loan_id,
                l.status,
                d.region
            FROM loan l
            JOIN account a ON l.account_id = a.account_id
            JOIN target_districts d ON a.district_id = d.district_id
        )
        SELECT 
            region,
            CAST(SUM(CASE WHEN status IN ('B', 'D') THEN 1 ELSE 0 END) AS REAL) / COUNT(*) AS average_default_rate
        FROM loans_in_regions
        GROUP BY region
    """,

    # [CR13] (regional_risk, moderate)
    # Question: Which districts have a default rate above 10%?
    # Evidence: Default rate = defaulted loans / total loans per district. Default means status B or D.
    "CR13": """
        SELECT 
            d.district_id,
            d.A2 AS district_name,
            CAST(SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) AS REAL) / COUNT(l.loan_id) AS default_rate
        FROM district d
        JOIN account a ON d.district_id = a.district_id
        JOIN loan l ON a.account_id = l.account_id
        GROUP BY d.district_id, d.A2
        HAVING default_rate > 0.10
    """,

    # [CR07] (regional_risk, simple)
    # Question: Which region has the highest number of defaulted loan accounts?
    # Evidence: Default means status = 'D'. Region information is in district table column A3.
    "CR07": """
        SELECT 
            d.A3 AS region,
            COUNT(*) AS default_count
        FROM loan l
        JOIN account a ON l.account_id = a.account_id
        JOIN district d ON a.district_id = d.district_id
        WHERE l.status = 'D'
        GROUP BY d.A3
        ORDER BY default_count DESC
        LIMIT 1
    """,

    # [CR23] (transaction_behavior, challenging)
    # Question: Calculate the average balance trend (difference between last and first recorded balance) for accounts with defaulted loans versus non-defaulted loans.
    # Evidence: Default means status B or D. Balance trend = last balance - first balance.
    # 대체 형태 반영: MIN/MAX 서브쿼리를 이용해 계좌별 첫 거래와 마지막 거래의 트렌드를 안정적으로 추출하는 형태
    "CR23": """
        WITH first_last_ids AS (
            SELECT 
                account_id,
                (SELECT trans_id FROM trans t2 WHERE t2.account_id = t1.account_id ORDER BY date ASC, trans_id ASC LIMIT 1) AS first_id,
                (SELECT trans_id FROM trans t3 WHERE t3.account_id = t1.account_id ORDER BY date DESC, trans_id DESC LIMIT 1) AS last_id
            FROM trans t1
            GROUP BY account_id
        ),
        trends AS (
            SELECT 
                f.account_id,
                (t_last.balance - t_first.balance) AS balance_trend
            FROM first_last_ids f
            JOIN trans t_first ON f.first_id = t_first.trans_id
            JOIN trans t_last ON f.last_id = t_last.trans_id
        )
        SELECT 
            CASE WHEN l.status IN ('B', 'D') THEN 'Defaulted' ELSE 'Non-Defaulted' END AS loan_group,
            AVG(tr.balance_trend) AS avg_balance_trend
        FROM trends tr
        JOIN loan l ON tr.account_id = l.account_id
        GROUP BY loan_group
    """,

    # [CR16] (transaction_behavior, moderate)
    # Question: How many accounts had a negative balance at any point in their transaction history?
    # Evidence: Negative balance means balance < 0 in the trans table.
    "CR16": """
        SELECT COUNT(DISTINCT account_id) AS negative_balance_accounts_count
        FROM trans
        WHERE balance < 0
    """,

    # [CR08] (transaction_behavior, simple)
    # Question: What is the average account balance across all accounts?
    # Evidence: Balance is recorded in the trans table after each transaction.
    # 대체 형태 반영: trans 테이블 내 모든 트랜잭션 행의 단순 평균 계산 방식
    "CR08": """
        SELECT AVG(balance) AS avg_account_balance
        FROM trans
    """
}

print(f"You have written SQL for {sum(1 for v in my_sql.values() if v)} "
      f"of {len(sample)} sampled questions.")
missing = [q['question_id'] for q in sample if not my_sql.get(q['question_id'])]
if missing:
    print(f"Not yet written: {missing}")

SCRIPT VERSION: 2026-07-15-v1
DB_PATH in use: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
Loaded 60 total questions.
Stratified sample size: 18
question_id category               difficulty  
CR27     card_risk              challenging 
CR19     card_risk              moderate    
CR58     card_risk              simple      
CR55     client_profile         challenging 
CR18     client_profile         moderate    
CR50     client_profile         simple      
CR21     default_risk           challenging 
CR32     default_risk           moderate    
CR01     default_risk           simple      
CR25     loan_portfolio         challenging 
CR47     loan_portfolio         moderate    
CR45     loan_portfolio         simple      
CR24     regional_risk          challenging 
CR13     regional_risk          moderate    
CR07     regional_risk          simple      
CR23     transaction_behavior   challenging 
CR16     transaction_behavior   moderate    
CR08     tran

In [6]:
# %% [markdown]
# ## 3. Reveal gold SQL and compare (execution-based, not text-based)
#
# Both queries are EXECUTED against the live database and their
# result sets compared -- this catches logical equivalence/divergence
# that a text-only comparison would miss.

# %%
def execute_sql(sql):
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql(sql, conn)
        conn.close()
        return df, None
    except Exception as e:
        return None, str(e)

def normalize_result(df):
    if df is None or df.empty:
        return set()
    df = df.copy()
    df.columns = range(len(df.columns))
    df = df.map(lambda x: str(round(float(x), 4)) if isinstance(x, float) else str(x))
    return set(df.apply(tuple, axis=1))

validation_results = []

print("=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

sample_by_id = {q['question_id']: q for q in sample}

for qid, my_query in my_sql.items():
    if not my_query:
        continue
    q = sample_by_id.get(qid)
    if q is None:
        print(f"\n[{qid}] !! not found in this sample -- skipping "
              f"(check for a typo in the question_id)")
        continue

    gold_sql = q['SQL']

    df_mine, err_mine = execute_sql(my_query)
    df_gold, err_gold = execute_sql(gold_sql)

    print(f"\n[{qid}] ({q['category']}, {q['difficulty']})")
    print(f"  Question: {q['question']}")
    print(f"  Your SQL:  {my_query}")
    print(f"  Gold SQL:  {gold_sql}")

    if err_mine:
        print(f"  !! Your SQL failed to execute: {err_mine}")
        status = "your_sql_error"
    elif err_gold:
        print(f"  !! Gold SQL failed to execute: {err_gold}")
        status = "gold_sql_error"
    else:
        match = normalize_result(df_mine) == normalize_result(df_gold)
        status = "match" if match else "MISMATCH"
        print(f"  Your result:\n{df_mine.to_string(index=False, max_rows=5)}")
        print(f"  Gold result:\n{df_gold.to_string(index=False, max_rows=5)}")
        print(f"  >>> {'✓ MATCH' if match else '✗ MISMATCH -- inspect manually'}")

    validation_results.append({
        'question_id': qid, 'category': q['category'], 'difficulty': q['difficulty'],
        'question': q['question'], 'my_sql': my_query, 'gold_sql': gold_sql,
        'status': status,
    })

# %% [markdown]
# ## 4. Summary table for the manuscript's provenance statement

# %%
df_validation = pd.DataFrame(validation_results)

if len(df_validation) == 0:
    print("No questions were validated (my_sql was empty). Complete")
    print("Section 2 and re-run Section 3 before generating a summary.")
else:
    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(df_validation['status'].value_counts())

    n_match = (df_validation['status'] == 'match').sum()
    n_total = len(df_validation)

    print(f"\n{n_match} / {n_total} independently-written queries produced")
    print(f"identical execution results to the corresponding gold SQL.")

    mismatches = df_validation[df_validation['status'] != 'match']
    if len(mismatches) > 0:
        print(f"\n{len(mismatches)} question(s) require manual review:")
        for _, row in mismatches.iterrows():
            print(f"  [{row['question_id']}] status={row['status']}")
            print(f"    my_sql:   {row['my_sql']}")
            print(f"    gold_sql: {row['gold_sql']}")

    print("\n" + "=" * 70)
    print("SUGGESTED MANUSCRIPT TEXT")
    print("=" * 70)
    print(f"""
"As an additional sanity check, the sole author independently
rewrote SQL for a stratified sample of {n_total} of the 60 benchmark
questions (one question per category-difficulty combination),
without reference to the original gold query, based solely on the
question text, accompanying evidence, and the semantic layer schema.
{n_match} of {n_total} independently-written queries produced results
identical to the corresponding gold query when both were executed
against the live database{'.' if len(mismatches) == 0 else f', and the remaining {len(mismatches)} were reviewed and [describe actual resolution here].'}"
""")

    df_validation.to_csv('gold_sql_independent_validation.csv', index=False, encoding='utf-8-sig')
    print("\nSaved full validation record to: gold_sql_independent_validation.csv")
    print("(suitable for inclusion as supplementary material)")

VALIDATION RESULTS

[CR27] (card_risk, challenging)
  Question: Among accounts with gold credit cards, what proportion have loan contracts in default?
  Your SQL:  
        SELECT 
            CAST(SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) AS REAL) / COUNT(DISTINCT a.account_id) AS proportion
        FROM card c
        JOIN disp d ON c.disp_id = d.disp_id
        JOIN account a ON d.account_id = a.account_id
        JOIN loan l ON a.account_id = l.account_id
        WHERE c.type = 'gold'
    
  Gold SQL:  SELECT CAST(SUM(CASE WHEN loan.status IN ('B','D') THEN 1 ELSE 0 END) AS REAL) / COUNT(DISTINCT account.account_id) * 100 AS default_rate FROM account JOIN disp ON account.account_id = disp.account_id JOIN card ON disp.disp_id = card.disp_id JOIN loan ON account.account_id = loan.account_id WHERE card.type = 'gold'
  Your result:
 proportion
     0.0625
  Gold result:
 default_rate
         6.25
  >>> ✗ MISMATCH -- inspect manually

[CR19] (card_risk, moderate)
  Questi

In [5]:
# %% [markdown]
# # 17_cr41_revalidation.ipynb
# Focused re-validation of CR41 ONLY, after its evidence was
# clarified (the "ratio" was ambiguous between a count-based and an
# amount-based interpretation; evidence now specifies count-based).
#
# WORKFLOW:
#   1. Run Section 1. It shows ONLY CR41's (updated) question and
#      evidence -- NOT the gold SQL.
#   2. Write your own SQL in Section 2's MY_CR41_SQL variable, based
#      solely on what Section 1 showed you.
#   3. Run Section 3. It reveals CR41's gold SQL and compares both
#      queries by EXECUTING them against the live database.
#   4. Run Section 4 to append the result to
#      gold_sql_independent_validation.csv (creates the file if it
#      doesn't exist yet, or updates the CR41 row if it does).

# %%
import json
import sqlite3
import pandas as pd
import os
from utils import DB_PATH

print(f"DB_PATH: {DB_PATH}")

# %% [markdown]
# ## 1. Show CR41's question and evidence ONLY (no gold SQL)

# %%
with open('credit_rating_questions_all.json', 'r', encoding='utf-8') as f:
    all_questions = json.load(f)

questions_by_id = {q['question_id']: q for q in all_questions}

if 'CR41' not in questions_by_id:
    raise KeyError("CR41 not found in credit_rating_questions_all.json")

cr41 = questions_by_id['CR41']

print("=" * 70)
print("CR41 -- write your SQL based ONLY on what's shown below")
print("=" * 70)
print(f"\nCategory:   {cr41.get('category', 'N/A')}")
print(f"Difficulty: {cr41.get('difficulty', 'N/A')}")
print(f"\nQuestion:\n  {cr41['question']}")
print(f"\nEvidence:\n  {cr41['evidence']}")

print("""

Schema reference (physical column names, for your own SQL writing):
  trans(trans_id, account_id, date, type, operation, amount, balance,
        k_symbol, bank, account)
  loan(loan_id, account_id, date, amount, duration, payments, status)

Do NOT scroll to Section 3 (which reveals the gold SQL) until you've
written your SQL in Section 2 below.
""")

# %% [markdown]
# ## 2. Write your independent SQL here

# %%
MY_CR41_SQL = """
SELECT 
    CAST(SUM(CASE WHEN t.type = 'VYDAJ' THEN 1 ELSE 0 END) AS REAL) 
    / NULLIF(SUM(CASE WHEN t.type = 'PRIJEM' THEN 1 ELSE 0 END), 0) AS withdrawal_deposit_ratio
FROM trans t
JOIN loan l ON t.account_id = l.account_id
WHERE l.status = 'D'
"""

if MY_CR41_SQL.strip().startswith('--'):
    print("!! MY_CR41_SQL still contains only the placeholder comment.")
    print("    Replace it with your actual SQL before running Section 3.")
else:
    print("Your SQL:")
    print(MY_CR41_SQL)

DB_PATH: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
CR41 -- write your SQL based ONLY on what's shown below

Category:   transaction_behavior
Difficulty: moderate

Question:
  For accounts with defaulted loans, what is the ratio of withdrawal transactions to deposit transactions?

Evidence:
  Withdrawal means type = 'VYDAJ'. Deposit means type = 'PRIJEM'. Default means loan status D. Ratio here refers to the NUMBER (count) of transactions, not their total monetary amount.


Schema reference (physical column names, for your own SQL writing):
  trans(trans_id, account_id, date, type, operation, amount, balance,
        k_symbol, bank, account)
  loan(loan_id, account_id, date, amount, duration, payments, status)

Do NOT scroll to Section 3 (which reveals the gold SQL) until you've
written your SQL in Section 2 below.

Your SQL:

SELECT 
    CAST(SUM(CASE WHEN t.type = 'VYDAJ' THEN 1 ELSE 0 END) AS REAL) 
    / NULLIF(SUM(CASE WHEN t.type = 'PRIJEM' THEN 1 E

In [6]:
# %% [markdown]
# ## 3. Reveal gold SQL and compare (execution-based)

# %%
cr41_gold_sql = cr41['SQL']

def execute_sql(sql, label):
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql(sql, conn)
        conn.close()
        print(f"  [{label}] SUCCESS -- {len(df)} row(s)")
        return df, None
    except Exception as e:
        print(f"  [{label}] FAILED: {e}")
        return None, str(e)

print("=" * 70)
print("CR41 comparison")
print("=" * 70)
print(f"Your SQL: {MY_CR41_SQL.strip()}")
print(f"Gold SQL: {cr41_gold_sql}")
print()

df_mine, err_mine = execute_sql(MY_CR41_SQL, "yours")
df_gold, err_gold = execute_sql(cr41_gold_sql, "gold")

status = None

if err_mine:
    print(f"\n>>> Your SQL failed to execute: {err_mine}")
    status = "your_sql_error"
elif err_gold:
    print(f"\n>>> Gold SQL failed to execute: {err_gold}")
    status = "gold_sql_error"
else:
    print("\n--- Your result ---")
    print(df_mine)
    print("\n--- Gold result ---")
    print(df_gold)

    def normalize_result(df):
        if df is None or df.empty:
            return set()
        df = df.copy()
        df.columns = range(len(df.columns))
        df = df.map(lambda x: str(round(float(x), 4)) if isinstance(x, float) else str(x))
        return set(df.apply(tuple, axis=1))

    match = normalize_result(df_mine) == normalize_result(df_gold)
    status = "match" if match else "MISMATCH"

    if len(df_mine) == 1 and len(df_gold) == 1 and len(df_mine.columns) == 1 and len(df_gold.columns) == 1:
        try:
            v_mine = float(df_mine.iloc[0, 0])
            v_gold = float(df_gold.iloc[0, 0])
            print(f"\nSingle-value comparison: yours={v_mine}, gold={v_gold}, "
                  f"diff={v_mine - v_gold:+.4f}")
        except (ValueError, TypeError):
            pass

    print(f"\n>>> {'✓ MATCH' if match else '✗ MISMATCH'}")

# %% [markdown]
# ## 4. Save/update the result in gold_sql_independent_validation.csv

# %%
new_row = {
    'question_id': 'CR41',
    'category': cr41.get('category', ''),
    'difficulty': cr41.get('difficulty', ''),
    'question': cr41['question'],
    'my_sql': MY_CR41_SQL.strip(),
    'gold_sql': cr41_gold_sql,
    'status': status,
}

CSV_PATH = 'gold_sql_independent_validation.csv'

if os.path.exists(CSV_PATH):
    df_existing = pd.read_csv(CSV_PATH)
    df_existing = df_existing[df_existing['question_id'] != 'CR41']
    df_updated = pd.concat([df_existing, pd.DataFrame([new_row])], ignore_index=True)
else:
    df_updated = pd.DataFrame([new_row])

df_updated.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f"Saved to {CSV_PATH}. Full contents:")
print(df_updated[['question_id', 'status']].to_string(index=False))

print(f"\nCR41 final status: {status}")
if status == 'match':
    print("✓ Re-validation confirms: after clarifying the evidence,")
    print("  the independently-written SQL now matches gold SQL.")
elif status == 'MISMATCH':
    print("✗ Still a mismatch even after evidence clarification --")
    print("  inspect the two result sets above to understand why.")

CR41 comparison
Your SQL: SELECT 
    CAST(SUM(CASE WHEN t.type = 'VYDAJ' THEN 1 ELSE 0 END) AS REAL) 
    / NULLIF(SUM(CASE WHEN t.type = 'PRIJEM' THEN 1 ELSE 0 END), 0) AS withdrawal_deposit_ratio
FROM trans t
JOIN loan l ON t.account_id = l.account_id
WHERE l.status = 'D'
Gold SQL: SELECT CAST(SUM(CASE WHEN trans.type = 'VYDAJ' THEN 1 ELSE 0 END) AS REAL) / NULLIF(SUM(CASE WHEN trans.type = 'PRIJEM' THEN 1 ELSE 0 END), 0) AS withdrawal_deposit_ratio FROM trans JOIN loan ON trans.account_id = loan.account_id WHERE loan.status = 'D'

  [yours] SUCCESS -- 1 row(s)
  [gold] SUCCESS -- 1 row(s)

--- Your result ---
   withdrawal_deposit_ratio
0                  1.387859

--- Gold result ---
   withdrawal_deposit_ratio
0                  1.387859

Single-value comparison: yours=1.387858719646799, gold=1.387858719646799, diff=+0.0000

>>> ✓ MATCH
Saved to gold_sql_independent_validation.csv. Full contents:
question_id status
       CR41  match

CR41 final status: match
✓ Re-validation conf